# Notebook 08 — Text2Cypher (a deliberate-failure demo)

> **Warning — this notebook is intentionally broken.** It exists to *show* what happens when you wire an LLM directly to a Cypher endpoint with nothing but the schema and a natural-language question. Do **not** copy this pattern into the production agent. The agent uses Pattern 2 (tool-bound Cypher: app code writes the query, the LLM only picks the tool + arguments) and Pattern 3 (hybrid vector + graph retrieval) precisely *because* this notebook fails.

## What is "Pattern 1"?

There are roughly three patterns for using an LLM against a knowledge graph:

1. **Pattern 1 — Text2Cypher.** Hand the LLM the schema and the user's question. Ask it to generate Cypher. Run that Cypher. Return the result. ← *This notebook.*
2. **Pattern 2 — Tool-bound Cypher.** App code holds a fixed set of parameterised Cypher queries; the LLM picks a tool by name and supplies arguments. The LLM never writes Cypher. See `src/va_agent/graph/tools.py` and the `cfr_*`/`user_*` split in `GraphDriver`.
3. **Pattern 3 — Hybrid retrieval.** Vector-search a text-bearing node (here: `:Criterion.embedding`), then graph-traverse from the hit to gather neighbours (Rating Levels, Diagnostic Codes, Measurements). See `src/va_agent/retrieval/matcher.py`.

Pattern 1 is seductive — it feels like "just talk to the graph." In practice it hallucinates labels, mis-reads operator semantics, and invents edges, and you have no good way to detect failure short of executing the query and squinting at the result. This notebook produces three such failures on purpose.

## Why keep it in the repo?

Per the PRD ([issue #1](https://github.com/leighton-tidwell/va-disability-agent/issues/1)) and tracked in [issue #12](https://github.com/leighton-tidwell/va-disability-agent/issues/12), Pattern 1 is *reserved as a demo cell* — a pedagogical artifact, never reachable from the agent's runtime. Reading the failures here is the cheapest way to internalise why production Cypher belongs in app code, not in a prompt.

## Prereqs

- `make up` — Neo4j running
- `OPENAI_API_KEY` in `.env`
- The CFR graph populated (run notebook 01 / 02 first so DC 5260, RatingLevels, Criteria, Measurements exist)
- **Do not execute this notebook expecting deterministic outputs.** Outputs depend on the LLM run; the *shape* of the failures is the lesson, not any specific generated Cypher string.

## 1. Setup — minimal Text2Cypher harness

We deliberately do not use `langchain-neo4j`'s `GraphCypherQAChain` (not a project dependency, and using it would obscure what's happening). Instead we hand-roll the same idea in ~30 lines: introspect schema → stuff it into a system prompt → ask `ChatOpenAI` for Cypher → execute via `GraphDriver.cfr_read`.

Note: no vector index, no embeddings. Text2Cypher's entire context for the LLM is the schema text. That's the point — Pattern 1 *cannot* see the actual data, only the shape.

In [ ]:
from textwrap import dedent

from langchain_openai import ChatOpenAI

from va_agent.graph.driver import GraphDriver

driver = GraphDriver.from_env()
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)


def introspect_schema(driver: GraphDriver) -> str:
    """Return a compact textual schema description for the LLM prompt.

    Just labels, relationship types, and property keys. No samples, no
    explanations — the same thing GraphCypherQAChain would feed it.
    """
    labels = [r["label"] for r in driver.cfr_read("CALL db.labels() YIELD label RETURN label")]
    rels = [r["relationshipType"] for r in driver.cfr_read(
        "CALL db.relationshipTypes() YIELD relationshipType RETURN relationshipType"
    )]
    props = [r["propertyKey"] for r in driver.cfr_read(
        "CALL db.propertyKeys() YIELD propertyKey RETURN propertyKey"
    )]
    return dedent(f"""\
        Node labels: {sorted(labels)}
        Relationship types: {sorted(rels)}
        Property keys: {sorted(props)}
    """)


SYSTEM_TEMPLATE = dedent("""\
    You are a Cypher-writing assistant for a Neo4j 5 graph of US VA disability
    rating schedule data (38 CFR Part 4). Given the schema and a question,
    output ONLY a single Cypher query — no prose, no markdown fences. The
    query must be safe to execute as a read-only statement.

    Schema:
    {schema}
""")


def text2cypher(question: str, schema: str) -> str:
    msgs = [
        ("system", SYSTEM_TEMPLATE.format(schema=schema)),
        ("user", question),
    ]
    resp = llm.invoke(msgs)
    text = resp.content.strip()
    # Strip code fences if the LLM ignored instructions (common).
    if text.startswith("```"):
        text = text.strip("`")
        if text.lower().startswith("cypher"):
            text = text[len("cypher"):]
        text = text.strip()
    return text


def run_demo(question: str, schema: str) -> None:
    print(f"Q: {question}\n")
    cypher = text2cypher(question, schema)
    print("LLM-generated Cypher:")
    print(cypher)
    print()
    try:
        rows = driver.cfr_read(cypher)
        print(f"Result ({len(rows)} rows):")
        for r in rows[:10]:
            print("  ", r)
        if len(rows) > 10:
            print(f"  ... ({len(rows) - 10} more)")
    except Exception as exc:
        print(f"Cypher execution error: {exc!r}")

## 2. The schema the LLM gets

Before we ask any questions, look at what the LLM has to work with. This is the *entire* context Pattern 1 gives it — no examples, no documentation of what a `:Criterion.text` looks like, no hint that `Measurement.operator` is a string like `"<="` rather than a Cypher operator. The LLM must infer everything from these names.

After the populated graph (notebooks 01–02 plus ingestion of more sections), expect to see labels including `CFR`, `DiagnosticCode`, `RatingLevel`, `Criterion`, `Measurement`, `Anatomy`, `Note`, `CrossReference`, `Rule`, `Section` (and user-side labels like `Veteran`, `User`, `Symptom`, `Measurement` if a tracer persona has been recorded). Relationship types include `HAS_RATING`, `REQUIRES`, `CRITERION_FOR`, `HAS_MEASUREMENT`, `OF_ANATOMY`, `CROSS_REFERENCES`, `HAS_NOTE`, `IN_SECTION`.

In [ ]:
schema = introspect_schema(driver)
print(schema)

## 3. Failure mode A — hallucinated labels

Ask a question whose subject doesn't exist in the v1 graph. There is no `:Claim` node, no `:Veteran-Claim` edge, and `:Veteran` only exists after a tracer-persona run (notebook 02). The LLM has a strong prior — "VA disability" → "veterans" → "claims" — and will happily invent labels and relationships that *sound* right.

Watch for: invented labels (`:Filing`, `:Application`), invented relationships (`:FILED`, `:SUBMITTED`), or correct labels combined with non-existent properties (`v.claim_count`). The query may run and return `0` (worse than an error — looks like a real answer).

In [ ]:
run_demo(
    "How many veterans have filed claims?",
    schema,
)

**What went wrong (expected).** "Filed a claim" isn't in the schema at all — Pattern 1 has no way to tell the LLM that the v1 graph models CFR text, not user filings. The LLM hallucinates a `:Claim` label (or `:Filing`, or a `:FILED` edge), Cypher dutifully matches zero rows, and the user gets a confident `0`. There is no signal anywhere in the chain that the *question* was unanswerable; the failure is silent.

In production this is the worst failure mode: a wrong-but-clean number that the LLM will then *narrate* as fact.

## 4. Failure mode B — wrong-but-plausible data

Now ask something the graph *can* answer, but whose answer depends on subtle semantics the schema does not encode.

The Measurement nodes carry `operator` (a string like `"<="`, `">="`, `"="`) and `value` and `unit`. The CFR semantics are: *the veteran's measured value must satisfy the criterion's operator vs. value to qualify for that Rating Level.* A 30° flexion satisfies `<=45`, which is the criterion for the **30% level** under DC 5260. The 30% level requires the *more restricted* flexion — i.e. the lowest threshold value, not the highest.

Watch for: the LLM picking the *largest* value to mean "highest rating" (it's the opposite); ignoring the operator entirely ("min knee flexion" → `MIN(m.value)` which mixes ≤ and = thresholds); or returning a value from a different Diagnostic Code because it forgot to scope by DC.

In [ ]:
run_demo(
    "What's the minimum knee flexion in degrees for the highest rating under DC 5260?",
    schema,
)

**What went wrong (expected).** The Cypher likely runs and returns *a number*. That number is almost certainly wrong for at least one of three reasons:

1. "Highest rating" was interpreted as `MAX(rl.percent)` — fine — but "minimum knee flexion" was interpreted as `MIN(m.value)` across all measurements on that level. Since the Measurement may carry a `<=` operator, the *minimum threshold value* is what qualifies *more* veterans, not fewer; the LLM has no way to know that without reading §4.71a prose.
2. The LLM may not have filtered by anatomy = "knee" and pulled in a `Measurement` attached to a different criterion.
3. Operator semantics (`"<="` is a *string*, not a Cypher comparator) are routinely flattened by the LLM into a numeric comparison, which Cypher executes silently against the wrong column.

A numeric answer that looks plausible is the second-worst failure mode: *almost-right* answers train the user to trust the pipeline.

## 5. Failure mode C — query runs, returns nonsense

Ask about a *concept* that lives in the CFR (and in `CONTEXT.md`) but is **not a property or label** in the graph. Pyramiding (§4.14) is a Rule — it's stored as a `:Rule` node with `id = "4.14"` and the rule text in `r.text`. There is no `dc.pyramids` boolean and no `:PYRAMIDS_WITH` edge. The LLM, fed only labels and property keys, will not see `pyramiding` *anywhere* in the schema and will invent something.

Watch for: a `WHERE dc.pyramiding = true` clause matching no rows; a `[:PYRAMIDS_WITH]` traversal failing silently; or a creative `WHERE dc.title CONTAINS "pyramid"` returning the empty set.

In [ ]:
run_demo(
    "List every Diagnostic Code that pyramids with another.",
    schema,
)

**What went wrong (expected).** Pyramiding is *the rule* — a single `:Rule` node — that constrains how *any* pair of overlapping Diagnostic Codes is rated. It is not a per-DC property. The correct answer requires reading the rule text and reasoning about symptom overlap across DCs; no Cypher query, generated or hand-written, can answer this in one shot against the current schema.

Pattern 1 cannot represent "this question is the wrong shape for the graph." It can only emit Cypher, run it, and return whatever rows come back. In this case: an empty list with the same confidence as a real answer.

## 6. What this teaches

- **App code must own Cypher.** In production the LLM picks a tool by name and supplies typed arguments; the Cypher is parameterised and lives in `src/va_agent/graph/tools.py`. That is Pattern 2. The split between `cfr_read` and `user_read`/`user_write` in `GraphDriver` is the structural guarantee — an LLM-emitted argument cannot escape its user scope or its read/write boundary.
- **Text retrieval belongs on text.** Questions about *what does the CFR say about X* should go through Pattern 3 — vector search over `:Criterion.text` (and `:Rule.text` once embedded), then graph traversal from the hit. The Match Retriever (`src/va_agent/retrieval/matcher.py`) is the canonical example. The LLM never sees Cypher; it sees structured candidate matches.
- **Schema-only context is not enough.** Pattern 1 fails because the schema names (`Measurement.operator`, `RatingLevel.percent`) do not carry their semantics. Those semantics live in §4.71a prose and in `CONTEXT.md` — and neither is in the prompt.
- **Where Pattern 1 *is* fine.** Read-only exploratory queries on a graph you understand, with a human reviewing every generated Cypher before it executes. "Explain my data to me" — not "answer my user's question."

## Cleanup

In [ ]:
driver.close()
print("Done.")